<a href="https://colab.research.google.com/github/dennisgathu8/36CHAMBERS/blob/main/Instagram_Content_Strategy_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas
!pip install plotly

In [9]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"
from google.colab import files

print("Please upload the file")
instagram_data = files.upload()
file1_name = list(instagram_data.keys())[0]
df1 = pd.read_csv(file1_name)
print(df1.head())

Please upload the file


Saving Instagram-Data.csv to Instagram-Data (3).csv
        Post ID                                        Description  \
0  1.799668e+16  Building end-to-end projects will help you lea...   
1  1.800604e+16  Data Engineers are crucial in ensuring that da...   
2  1.829417e+16  Here’s a list of 190+ Data Science projects ba...   
3  1.809139e+16  During cricket leagues like IPL, we see a lot ...   
4  1.796449e+16  The tools used by a data scientist can vary si...   

   Duration (secs)      Publish time  \
0                0  01/02/2024 20:19   
1                0  01/01/2024 20:26   
2                0  03/29/2024 23:00   
3                0  03/29/2024 02:52   
4                0  03/28/2024 02:50   

                                  Permalink    Post type  Data comment  \
0  https://www.instagram.com/p/C1n9wjkLL71/  IG carousel           NaN   
1  https://www.instagram.com/p/C1lZvITLD3O/  IG carousel           NaN   
2  https://www.instagram.com/p/C5IKcAfLurI/     IG image        

In [12]:
import plotly.express as px

# filter data for relevant metrics and post types
engagement_metrics = ['Impressions', 'Reach', 'Likes', 'Shares', 'Follows', 'Comments', 'Saves']
content_performance = df1[['Post type'] + engagement_metrics]

# group by 'Post type' and calculate average engagement metrics
performance_summary = content_performance.groupby('Post type').mean().reset_index()

# melt the data for easier plotting
performance_melted = performance_summary.melt(id_vars=['Post type'], var_name='Metric', value_name='Average Value')

# plotting engagement metrics by post type
fig = px.bar(
    performance_melted,
    x='Metric',
    y='Average Value',
    color='Post type',
    barmode='group',
    title='Average Engagement Metrics by Post Type',
    labels={'Metric': 'Engagement Metric', 'Average Value': 'Average Value'},
    text_auto='.2s'
)

fig.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    legend_title='Post Type',
    xaxis_tickangle=45
)

fig.show()

In [14]:
# extracting and converting publish time to datetime
df1['Publish time'] = pd.to_datetime(df1['Publish time'])
# creating additional time-based features
df1['Day of Week'] = df1['Publish time'].dt.day_name()
df1['Hour of Day'] = df1['Publish time'].dt.hour

time_analysis = df1[['Day of Week', 'Hour of Day', 'Impressions', 'Reach']]
# aggregating metrics
day_of_week_summary = time_analysis.groupby('Day of Week')[['Impressions', 'Reach']].mean().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
)
hour_of_day_summary = time_analysis.groupby('Hour of Day')[['Impressions', 'Reach']].mean()

fig_day = go.Figure()
fig_day.add_trace(go.Bar(
    x=day_of_week_summary.index,
    y=day_of_week_summary['Impressions'],
    name='Impressions',
    marker_color='steelblue'
))
fig_day.add_trace(go.Bar(
    x=day_of_week_summary.index,
    y=day_of_week_summary['Reach'],
    name='Reach',
    marker_color='orange'
))
fig_day.update_layout(
    title='Average Reach and Impressions by Day of the Week',
    xaxis_title='Day of the Week',
    yaxis_title='Average Value',
    barmode='group',
    xaxis_tickangle=45
)

fig_day.show()

# visualization: Hour of the Day
fig_hour = go.Figure()
fig_hour.add_trace(go.Scatter(
    x=hour_of_day_summary.index,
    y=hour_of_day_summary['Impressions'],
    mode='lines+markers',
    name='Impressions',
    line=dict(color='steelblue')
))
fig_hour.add_trace(go.Scatter(
    x=hour_of_day_summary.index,
    y=hour_of_day_summary['Reach'],
    mode='lines+markers',
    name='Reach',
    line=dict(color='orange')
))
fig_hour.update_layout(
    title='Average Reach and Impressions by Hour of the Day',
    xaxis_title='Hour of the Day',
    yaxis_title='Average Value',
    xaxis=dict(tickmode='linear', tick0=0, dtick=1)
)

fig_hour.show()

In [15]:
# clean the descriptions by removing hashtags
df1['Cleaned Description'] = df1['Description'].str.replace(r'#\S+', '', regex=True)
interaction_metrics = ['Likes', 'Comments', 'Saves']

# define topics and their associated keywords
topics = {
    "Projects": ["projects", "solved", "explained", "ideas"],
    "Learning and Education": ["learning", "science", "learn", "python", "analysis"],
    "Problem-Solving": ["problems", "using", "algorithms"],
    "Actionable Content": ["try", "use", "list", "let"],
    "Career Growth": ["link", "bio"],
}

# aggregate engagement metrics by topic
topic_engagement = {
    topic: df1[
        df1['Cleaned Description']
        .str.contains('|'.join(keywords), case=False, na=False)][interaction_metrics].mean()
    for topic, keywords in topics.items()
}

# convert to DataFrame
topic_engagement_df = pd.DataFrame(topic_engagement).T.reset_index()
topic_engagement_df.columns = ['Topic', 'Likes', 'Comments', 'Saves']

# melt DataFrame for Plotly
topic_engagement_melted = topic_engagement_df.melt(
    id_vars='Topic', var_name='Engagement Metric', value_name='Average Value'
)

fig = px.bar(
    topic_engagement_melted,
    x='Topic',
    y='Average Value',
    color='Engagement Metric',
    barmode='group',
    title='Average User Interactions by Topic',
    labels={'Average Value': 'Average Interactions'},
    text_auto='.2s',
)

fig.update_layout(
    title_font_size=16,
    xaxis_title='Topic',
    yaxis_title='Average Interactions',
    legend_title='Engagement Metric',
    xaxis_tickangle=45
)

fig.show()